In [2]:
!pip install astroquery

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.6 MB/s eta 0:00:00


In [4]:
"""
WILL Relational Orbital Mechanics (R.O.M.)
Empirical Verification: Juno Earth Flyby (Oct 9, 2013)
3D vector treatment of gravitational deflection.
"""

import numpy as np
from astroquery.jplhorizons import Horizons
from astropy.time import Time

# --- STRICT SI CONSTANTS (METERS & SECONDS ONLY) ---
C_M_S = 299792458.0           # Speed of light [m/s]
AU_M = 149597870700.0         # Astronomical Unit [m]
SEC_PER_DAY = 86400.0         # Seconds in a day [s]
R_S_EARTH_M = 0.008870056     # Schwarzschild radius of the Earth [m]


def get_state_vectors_si(target_id, center_id, epoch_str):
    """Return (position [m], velocity [m/s]) of target w.r.t. center at epoch."""
    jd = Time(epoch_str).jd
    obj = Horizons(id=target_id, location=center_id, epochs=jd)
    vec = obj.vectors()
    pos_m = np.array([vec['x'][0], vec['y'][0], vec['z'][0]]) * AU_M
    vel_ms = np.array([vec['vx'][0], vec['vy'][0], vec['vz'][0]]) * (AU_M / SEC_PER_DAY)
    return pos_m, vel_ms


def get_perigee_distance(target_id, center_id, start, stop, step):
    """Find the minimum geocentric distance over the given window [m]."""
    obj = Horizons(id=target_id, location=center_id,
                   epochs={'start': start, 'stop': stop, 'step': step})
    vec = obj.vectors()
    pos_m = np.column_stack((vec['x'], vec['y'], vec['z'])) * AU_M
    distances = np.linalg.norm(pos_m, axis=1)
    return float(np.min(distances))


def rodrigues_rotate(v, axis, angle):
    """
    Rotate vector v by `angle` radians around `axis` (right-hand rule).
    Rodrigues formula: v' = v cos(t) + (k x v) sin(t) + k (k.v) (1 - cos(t))
    """
    k = axis / np.linalg.norm(axis)
    return (v * np.cos(angle)
            + np.cross(k, v) * np.sin(angle)
            + k * np.dot(k, v) * (1.0 - np.cos(angle)))


def run_rom_deflection_3d():
    epoch_in = '2013-10-08 06:00:00'
    epoch_out = '2013-10-11 06:00:00'
    peri_start = '2013-10-09 18:00'
    peri_stop = '2013-10-09 21:00'

    print("Fetching JPL ephemeris (m, m/s)...")

    try:
        # Heliocentric Juno (incoming, for diagnostics)
        _, vel_juno_sun_in_ms = get_state_vectors_si('-61', '@10', epoch_in)
        # Heliocentric Juno (outgoing, observed truth)
        _, vel_juno_sun_out_obs_ms = get_state_vectors_si('-61', '@10', epoch_out)

        # Earth heliocentric at IN and OUT epochs.
        # Earth's velocity DIRECTION rotates by ~20 deg in 20 days — the
        # outgoing law of cosines must use the OUT-epoch direction.
        _, vel_earth_sun_in_ms = get_state_vectors_si('399', '@10', epoch_in)
        _, vel_earth_sun_out_ms = get_state_vectors_si('399', '@10', epoch_out)

        # Juno geocentric: position AND velocity at the IN epoch.
        # The position vector is required to fix the orbital plane normal
        # n_hat = (r x v) / |r x v|, which determines the deflection plane.
        pos_juno_earth_in_m, vel_juno_earth_in_ms = get_state_vectors_si('-61', '@399', epoch_in)

        # Perigee distance (magnitude only — needed for kappa_p^2)
        r_p_m = get_perigee_distance('-61', '@399', peri_start, peri_stop, '1m')

    except Exception as e:
        print(f"API Error: {e}")
        return

    # ==========================================
    # --- R.O.M. CORE (DIMENSIONLESS LOCAL FRAME) ---
    # ==========================================

    # Asymptotic incoming geocentric kinetic projection
    beta_inf = np.linalg.norm(vel_juno_earth_in_ms) / C_M_S

    # Gravitational projection at perigee (Earth's R_s and geocentric r_p)
    kappa_p_sq = R_S_EARTH_M / r_p_m

    # Energy invariant for an unbound (hyperbolic) orbit:
    #   kappa_o^2 - beta_o^2 = -beta_inf^2
    # at perigee:
    #   beta_p^2 = kappa_p^2 + beta_inf^2
    beta_p_sq = beta_inf**2 + kappa_p_sq

    # Closure factor and hyperbolic eccentricity
    delta_p = kappa_p_sq / (2.0 * beta_p_sq)
    e = (1.0 / delta_p) - 1.0

    # Asymptotic deflection angle for a hyperbolic flyby
    delta_phi = 2.0 * np.arcsin(1.0 / e)

    # ==========================================
    # --- 3D ROTATION IN THE ACTUAL ORBITAL PLANE ---
    # ==========================================

    # The orbital plane of the flyby is fixed by Juno's position and velocity
    # relative to Earth. Angular momentum direction (conserved along the orbit):
    #     L_hat = (r x v) / |r x v|     (right-hand rule)
    # The deflection rotates the asymptotic incoming velocity by delta_phi
    # in the orbital plane, around L_hat.
    L_vec = np.cross(pos_juno_earth_in_m, vel_juno_earth_in_ms)
    L_hat = L_vec / np.linalg.norm(L_vec)

    # Apply Rodrigues rotation to v_inf_in by delta_phi around L_hat.
    # (Magnitude is preserved: in Earth frame, |v_inf_out| = |v_inf_in|.)
    vel_juno_earth_out_ms = rodrigues_rotate(vel_juno_earth_in_ms, L_hat, delta_phi)

    # ==========================================
    # --- HELIOCENTRIC RECONSTRUCTION ---
    # ==========================================

    # Use Earth's heliocentric velocity at the OUT epoch (rotated direction).
    vel_juno_sun_out_pred_ms = vel_juno_earth_out_ms + vel_earth_sun_out_ms

    v_in_helio_obs_ms = np.linalg.norm(vel_juno_sun_in_ms)
    v_out_helio_obs_ms = np.linalg.norm(vel_juno_sun_out_obs_ms)
    v_out_helio_pred_ms = np.linalg.norm(vel_juno_sun_out_pred_ms)

    anomaly_ms = v_out_helio_obs_ms - v_out_helio_pred_ms
    anomaly_mms = anomaly_ms * 1000.0

    # ==========================================
    # --- DIAGNOSTIC OUTPUT ---
    # ==========================================

    print("\n--- R.O.M. 3D DEFLECTION RESULTS ---")
    print(f"Asymptotic geocentric speed v_inf:    {beta_inf * C_M_S / 1000.0:.4f} km/s")
    print(f"Perigee distance r_p:                 {r_p_m / 1000.0:.3f} km")
    print(f"kappa_p^2:                            {kappa_p_sq:.6e}")
    print(f"beta_p^2:                             {beta_p_sq:.6e}")
    print(f"Closure factor delta_p:               {delta_p:.6f}")
    print(f"Hyperbolic eccentricity e:            {e:.6f}")
    print(f"Deflection angle Delta_phi:           {np.degrees(delta_phi):.4f} deg")
    print(f"Orbital plane normal L_hat:           [{L_hat[0]:+.6f}, {L_hat[1]:+.6f}, {L_hat[2]:+.6f}]")
    print("---------------------------------------------------")
    print(f"Heliocentric speed in (JPL):          {v_in_helio_obs_ms / 1000.0:.6f} km/s")
    print(f"Heliocentric speed out, observed:     {v_out_helio_obs_ms / 1000.0:.6f} km/s")
    print(f"Heliocentric speed out, R.O.M. (3D):  {v_out_helio_pred_ms / 1000.0:.6f} km/s")
    print(f"Velocity anomaly (Observed - ROM):    {anomaly_mms:.4f} mm/s")


if __name__ == '__main__':
    run_rom_deflection_3d()

Fetching JPL ephemeris (m, m/s)...

--- R.O.M. 3D DEFLECTION RESULTS ---
Asymptotic geocentric speed v_inf:    10.4166 km/s
Perigee distance r_p:                 6941.816 km
kappa_p^2:                            1.277772e-09
beta_p^2:                             2.485062e-09
Closure factor delta_p:               0.257090
Hyperbolic eccentricity e:            2.889681
Deflection angle Delta_phi:           40.4929 deg
Orbital plane normal L_hat:           [+0.264050, -0.342000, +0.901839]
---------------------------------------------------
Heliocentric speed in (JPL):          35.060654 km/s
Heliocentric speed out, observed:     38.662653 km/s
Heliocentric speed out, R.O.M. (3D):  38.682825 km/s
Velocity anomaly (Observed - ROM):    -20171.8541 mm/s


In [3]:
"""
WILL Relational Orbital Mechanics (R.O.M.)
Empirical Verification: Residual Potential Boundary Compensation
Target: Juno Earth Flyby (Narrow Window: Oct 8 - Oct 11, 2013)
"""

import numpy as np
from astroquery.jplhorizons import Horizons
from astropy.time import Time

# --- STRICT SI CONSTANTS (METERS & SECONDS ONLY) ---
C_M_S = 299792458.0           # Speed of light [m/s]
AU_M = 149597870700.0         # Astronomical Unit [m]
SEC_PER_DAY = 86400.0         # Seconds in a day [s]
R_S_EARTH_M = 0.008870056     # Schwarzschild radius of the Earth [m]

def get_state_vectors_si(target_id, center_id, epoch_str):
    """Fetches state vectors and STRICTLY converts them to meters and meters/second."""
    jd = Time(epoch_str).jd
    obj = Horizons(id=target_id, location=center_id, epochs=jd)
    vec = obj.vectors()
    pos_m = np.array([vec['x'][0], vec['y'][0], vec['z'][0]]) * AU_M
    vel_ms = np.array([vec['vx'][0], vec['vy'][0], vec['vz'][0]]) * (AU_M / SEC_PER_DAY)
    return pos_m, vel_ms

def get_perigee_meters(target_id, center_id, start, stop, step):
    """Finds perigee distance strictly in meters."""
    obj = Horizons(id=target_id, location=center_id, epochs={'start': start, 'stop': stop, 'step': step})
    vec = obj.vectors()
    pos_m = np.column_stack((vec['x'], vec['y'], vec['z'])) * AU_M
    return float(np.min(np.linalg.norm(pos_m, axis=1)))

def rodrigues_rotate(v, axis, angle):
    """Rotates vector v by angle around normalized axis."""
    k = axis / np.linalg.norm(axis)
    return (v * np.cos(angle)
            + np.cross(k, v) * np.sin(angle)
            + k * np.dot(k, v) * (1.0 - np.cos(angle)))

def run_rom_boundary_compensation():
    # Narrow Window: Isolates local frame from the Sun's macro-gradient
    epoch_in = '2013-10-08 06:00:00'
    epoch_out = '2013-10-11 06:00:00'
    peri_start = '2013-10-09 18:00'
    peri_stop = '2013-10-09 21:00'

    print("Fetching JPL ephemeris (m, m/s) for Boundary Compensation...")

    try:
        # Local state vectors (Probe w.r.t Earth)
        pos_juno_earth_in_m, vel_juno_earth_in_ms = get_state_vectors_si('-61', '@399', epoch_in)
        pos_juno_earth_out_m, _ = get_state_vectors_si('-61', '@399', epoch_out)

        # Global state vectors (Probe and Earth w.r.t Sun at exit)
        _, vel_juno_sun_out_obs_ms = get_state_vectors_si('-61', '@10', epoch_out)
        _, vel_earth_sun_out_ms = get_state_vectors_si('399', '@10', epoch_out)

        r_p_m = get_perigee_meters('-61', '@399', peri_start, peri_stop, '1m')

    except Exception as e:
        print(f"API Error: {e}")
        return

    # =========================================================
    # --- 1. BOUNDARY COMPENSATION (TRUE ASYMPTOTE) ---
    # =========================================================

    r_in_m = np.linalg.norm(pos_juno_earth_in_m)
    r_out_m = np.linalg.norm(pos_juno_earth_out_m)

    # Residual potential capacity at observation boundaries
    kappa_E_in_sq = R_S_EARTH_M / r_in_m
    kappa_E_out_sq = R_S_EARTH_M / r_out_m

    # Raw observed local kinematics
    beta_loc_obs_in_sq = (np.linalg.norm(vel_juno_earth_in_ms) / C_M_S)**2

    # Lifting to True Infinity: Subtract boundary potential to find pure asymptote
    beta_inf_sq = beta_loc_obs_in_sq - kappa_E_in_sq

    if beta_inf_sq < 0:
        raise ValueError("Bound state detected. Not a flyby.")

    # =========================================================
    # --- 2. R.O.M. CORE (TOPOLOGICAL DEFECT) ---
    # =========================================================

    kappa_pE_sq = R_S_EARTH_M / r_p_m

    # Pure Vis-Viva equivalent using true infinity
    beta_pE_sq = beta_inf_sq + kappa_pE_sq

    delta_p = kappa_pE_sq / (2.0 * beta_pE_sq)
    e = (1.0 / delta_p) - 1.0
    delta_phi = 2.0 * np.arcsin(1.0 / e)

    # =========================================================
    # --- 3. 3D VECTOR ROTATION & BOUNDARY RESTORATION ---
    # =========================================================

    v_inf_ms = C_M_S * np.sqrt(beta_inf_sq)

    # Normalize incoming vector to serve as asymptotic direction
    dir_in = vel_juno_earth_in_ms / np.linalg.norm(vel_juno_earth_in_ms)
    vel_inf_in_ms = dir_in * v_inf_ms

    # Identify interaction plane normal
    L_vec = np.cross(pos_juno_earth_in_m, vel_juno_earth_in_ms)
    L_hat = L_vec / np.linalg.norm(L_vec)

    # Rotate the pure asymptotic vector
    vel_inf_out_ms = rodrigues_rotate(vel_inf_in_ms, L_hat, delta_phi)

    # Boundary Restoration: Drop the probe back into Earth's well to match epoch_out
    beta_loc_out_pred_sq = beta_inf_sq + kappa_E_out_sq
    v_loc_out_pred_ms = C_M_S * np.sqrt(beta_loc_out_pred_sq)

    dir_out = vel_inf_out_ms / np.linalg.norm(vel_inf_out_ms)
    vel_juno_earth_out_pred_ms = dir_out * v_loc_out_pred_ms

    # =========================================================
    # --- 4. GLOBAL MACRO-CLOSURE ---
    # =========================================================

    vel_juno_sun_out_pred_ms = vel_juno_earth_out_pred_ms + vel_earth_sun_out_ms

    v_out_pred_ms = np.linalg.norm(vel_juno_sun_out_pred_ms)
    v_out_obs_ms = np.linalg.norm(vel_juno_sun_out_obs_ms)

    anomaly_mms = (v_out_obs_ms - v_out_pred_ms) * 1000.0

    # --- TERMINAL OUTPUT ---
    print("\n--- R.O.M. BOUNDARY COMPENSATION RESULTS ---")
    print(f"Residual Potential IN (kappa_E_in^2):   {kappa_E_in_sq:.6e}")
    print(f"Residual Potential OUT (kappa_E_out^2): {kappa_E_out_sq:.6e}")
    print(f"True Asymptotic beta_inf^2:             {beta_inf_sq:.6e}")
    print(f"Topological Eccentricity (e):           {e:.6f}")
    print(f"Deflection Angle (Delta_phi):           {np.degrees(delta_phi):.4f} deg")
    print("---------------------------------------------------")
    print(f"Observed Terminal Velocity (JPL):       {v_out_obs_ms / 1000.0:.6f} km/s")
    print(f"Predicted Terminal Velocity (ROM):      {v_out_pred_ms / 1000.0:.6f} km/s")
    print("---------------------------------------------------")
    print(f"Velocity Anomaly (Observed - ROM):      {anomaly_mms:.4f} mm/s")

if __name__ == '__main__':
    run_rom_boundary_compensation()

Fetching JPL ephemeris (m, m/s) for Boundary Compensation...

--- R.O.M. BOUNDARY COMPENSATION RESULTS ---
Residual Potential IN (kappa_E_in^2):   6.269505e-12
Residual Potential OUT (kappa_E_out^2): 6.761657e-12
True Asymptotic beta_inf^2:             1.201021e-09
Topological Eccentricity (e):           2.879868
Deflection Angle (Delta_phi):           40.6370 deg
---------------------------------------------------
Observed Terminal Velocity (JPL):       38.662653 km/s
Predicted Terminal Velocity (ROM):      38.693540 km/s
---------------------------------------------------
Velocity Anomaly (Observed - ROM):      -30887.2029 mm/s


In [4]:
"""
WILL Relational Orbital Mechanics (R.O.M.)
Empirical Verification: Pure Topological Deflection Isolation
Target: Juno Earth Flyby (Oct 9, 2013)
"""

import numpy as np
from astroquery.jplhorizons import Horizons
from astropy.time import Time

# --- STRICT SI CONSTANTS (METERS & SECONDS ONLY) ---
C_M_S = 299792458.0           # Speed of light [m/s]
AU_M = 149597870700.0         # Astronomical Unit [m]
SEC_PER_DAY = 86400.0         # Seconds in a day [s]
R_S_EARTH_M = 0.008870056     # Schwarzschild radius of the Earth [m]

def get_state_vectors_si(target_id, center_id, epoch_str):
    jd = Time(epoch_str).jd
    obj = Horizons(id=target_id, location=center_id, epochs=jd)
    vec = obj.vectors()
    pos_m = np.array([vec['x'][0], vec['y'][0], vec['z'][0]]) * AU_M
    vel_ms = np.array([vec['vx'][0], vec['vy'][0], vec['vz'][0]]) * (AU_M / SEC_PER_DAY)
    return pos_m, vel_ms

def get_perigee_meters(target_id, center_id, start, stop, step):
    obj = Horizons(id=target_id, location=center_id, epochs={'start': start, 'stop': stop, 'step': step})
    vec = obj.vectors()
    pos_m = np.column_stack((vec['x'], vec['y'], vec['z'])) * AU_M
    return float(np.min(np.linalg.norm(pos_m, axis=1)))

def calculate_angle(v1, v2):
    unit_v1 = v1 / np.linalg.norm(v1)
    unit_v2 = v2 / np.linalg.norm(v2)
    return np.arccos(np.clip(np.dot(unit_v1, unit_v2), -1.0, 1.0))

def run_rom_topological_isolation():
    # WIDE WINDOW: True Asymptotes
    epoch_in = '2013-09-29 00:00:00'
    epoch_out = '2013-10-19 00:00:00'
    peri_start = '2013-10-09 18:00'
    peri_stop = '2013-10-09 21:00'

    print("Fetching JPL ephemeris (m, m/s) for Topological Isolation...")

    try:
        # Heliocentric
        _, vel_juno_sun_out_obs_ms = get_state_vectors_si('-61', '@10', epoch_out)
        _, vel_earth_sun_out_ms = get_state_vectors_si('399', '@10', epoch_out)

        # Geocentric
        pos_juno_earth_in_m, vel_juno_earth_in_ms = get_state_vectors_si('-61', '@399', epoch_in)
        _, vel_juno_earth_out_obs_ms = get_state_vectors_si('-61', '@399', epoch_out)

        r_p_m = get_perigee_meters('-61', '@399', peri_start, peri_stop, '1m')

    except Exception as e:
        print(f"API Error: {e}")
        return

    # =========================================================
    # --- 1. BASE PROJECTIONS (DIMENSIONLESS) ---
    # =========================================================

    beta_E_out = np.linalg.norm(vel_earth_sun_out_ms) / C_M_S
    beta_loc_in = np.linalg.norm(vel_juno_earth_in_ms) / C_M_S

    # Extracting exact local kinetics to isolate Sun's macro-gradient from Earth's phase rotation
    beta_loc_out_obs = np.linalg.norm(vel_juno_earth_out_obs_ms) / C_M_S

    # =========================================================
    # --- 2. R.O.M. TOPOLOGICAL DEFLECTION ---
    # =========================================================

    kappa_pE_sq = R_S_EARTH_M / r_p_m
    beta_pE_sq = (beta_loc_in**2) + kappa_pE_sq

    delta_p = kappa_pE_sq / (2.0 * beta_pE_sq)
    e = (1.0 / delta_p) - 1.0
    delta_phi = 2.0 * np.arcsin(1.0 / e)

    # =========================================================
    # --- 3. SPHERICAL PHASE CLOSURE (NO 3D ROTATIONS) ---
    # =========================================================

    # Reference Phase: Incoming local vs Outgoing Earth
    phi_ref = calculate_angle(vel_juno_earth_in_ms, vel_earth_sun_out_ms)

    # Topological Tilt (Dihedral Angle theta)
    n_orb = np.cross(pos_juno_earth_in_m, vel_juno_earth_in_ms)
    n_ref = np.cross(vel_juno_earth_in_ms, vel_earth_sun_out_ms)
    cos_theta = np.dot(n_orb, n_ref) / (np.linalg.norm(n_orb) * np.linalg.norm(n_ref))

    # Pure Spherical Cosine Law for Phase Shift
    cos_phi_out_pred = (np.cos(phi_ref) * np.cos(delta_phi)) + (np.sin(phi_ref) * np.sin(delta_phi) * cos_theta)
    phi_out_pred = np.arccos(cos_phi_out_pred)

    # =========================================================
    # --- 4. STRICT MACRO-ASSEMBLY ---
    # =========================================================

    # S1 Carrier Superposition using the Algebraically Predicted Phase
    beta_Z_out_pred_sq = (beta_E_out**2) + (beta_loc_out_obs**2) + (2.0 * beta_E_out * beta_loc_out_obs * cos_phi_out_pred)

    v_out_pred_ms = C_M_S * np.sqrt(beta_Z_out_pred_sq)
    v_out_obs_ms = np.linalg.norm(vel_juno_sun_out_obs_ms)

    anomaly_mms = (v_out_obs_ms - v_out_pred_ms) * 1000.0

    print("\n--- R.O.M. TOPOLOGICAL ISOLATION RESULTS ---")
    print(f"Topological Eccentricity (e):         {e:.6f}")
    print(f"Phase Deflection Arc (delta_phi):     {np.degrees(delta_phi):.4f} deg")
    print(f"Algebraic Outgoing Phase (phi_out):   {np.degrees(phi_out_pred):.4f} deg")
    print("---------------------------------------------------")
    print(f"Observed Terminal Velocity (JPL):     {v_out_obs_ms / 1000.0:.6f} km/s")
    print(f"Predicted Terminal Velocity (ROM):    {v_out_pred_ms / 1000.0:.6f} km/s")
    print("---------------------------------------------------")
    print(f"Velocity Anomaly (Observed - ROM):    {anomaly_mms:.4f} mm/s")

if __name__ == '__main__':
    run_rom_topological_isolation()

Fetching JPL ephemeris (m, m/s) for Topological Isolation...

--- R.O.M. TOPOLOGICAL ISOLATION RESULTS ---
Topological Eccentricity (e):         2.964090
Phase Deflection Arc (delta_phi):     39.4337 deg
Algebraic Outgoing Phase (phi_out):   40.5389 deg
---------------------------------------------------
Observed Terminal Velocity (JPL):     38.002692 km/s
Predicted Terminal Velocity (ROM):    38.399449 km/s
---------------------------------------------------
Velocity Anomaly (Observed - ROM):    -396756.6368 mm/s


In [5]:
"""
WILL Relational Orbital Mechanics (R.O.M.)
Empirical Verification: Direct Local Deflection Angle Check
Target: Juno Earth Flyby (Oct 9, 2013)
"""

import numpy as np
from astroquery.jplhorizons import Horizons
from astropy.time import Time

C_M_S = 299792458.0
AU_M = 149597870700.0
SEC_PER_DAY = 86400.0
R_S_EARTH_M = 0.008870056

def get_state_vectors_si(target_id, center_id, epoch_str):
    jd = Time(epoch_str).jd
    obj = Horizons(id=target_id, location=center_id, epochs=jd)
    vec = obj.vectors()
    pos_m = np.array([vec['x'][0], vec['y'][0], vec['z'][0]]) * AU_M
    vel_ms = np.array([vec['vx'][0], vec['vy'][0], vec['vz'][0]]) * (AU_M / SEC_PER_DAY)
    return pos_m, vel_ms

def get_perigee_meters(target_id, center_id, start, stop, step):
    obj = Horizons(id=target_id, location=center_id, epochs={'start': start, 'stop': stop, 'step': step})
    vec = obj.vectors()
    pos_m = np.column_stack((vec['x'], vec['y'], vec['z'])) * AU_M
    return float(np.min(np.linalg.norm(pos_m, axis=1)))

def calculate_angle(v1, v2):
    unit_v1 = v1 / np.linalg.norm(v1)
    unit_v2 = v2 / np.linalg.norm(v2)
    return np.arccos(np.clip(np.dot(unit_v1, unit_v2), -1.0, 1.0))

def run_rom_direct_angle_check():
    # WIDE WINDOW: True Asymptotes
    epoch_in = '2013-09-29 00:00:00'
    epoch_out = '2013-10-19 00:00:00'
    peri_start = '2013-10-09 18:00'
    peri_stop = '2013-10-09 21:00'

    print("Fetching JPL ephemeris (m, m/s) for Direct Angle Check...")

    try:
        # GEOCENTRIC ONLY (Probe w.r.t Earth)
        _, vel_juno_earth_in_ms = get_state_vectors_si('-61', '@399', epoch_in)
        _, vel_juno_earth_out_obs_ms = get_state_vectors_si('-61', '@399', epoch_out)

        r_p_m = get_perigee_meters('-61', '@399', peri_start, peri_stop, '1m')

    except Exception as e:
        print(f"API Error: {e}")
        return

    # 1. R.O.M. ALGEBRAIC PREDICTION
    beta_loc_in = np.linalg.norm(vel_juno_earth_in_ms) / C_M_S
    kappa_pE_sq = R_S_EARTH_M / r_p_m
    beta_pE_sq = (beta_loc_in**2) + kappa_pE_sq

    delta_p = kappa_pE_sq / (2.0 * beta_pE_sq)
    e = (1.0 / delta_p) - 1.0

    # Pure algebraic topological deflection
    delta_phi_rom_rad = 2.0 * np.arcsin(1.0 / e)
    delta_phi_rom_deg = np.degrees(delta_phi_rom_rad)

    # 2. JPL OBSERVED DEFLECTION (Ground Truth)
    # The actual angle between incoming and outgoing geocentric asymptotes
    delta_phi_obs_rad = calculate_angle(vel_juno_earth_in_ms, vel_juno_earth_out_obs_ms)
    delta_phi_obs_deg = np.degrees(delta_phi_obs_rad)

    # 3. ANALYSIS
    angle_anomaly_deg = delta_phi_obs_deg - delta_phi_rom_deg

    print("\n--- R.O.M. PURE LOCAL DEFLECTION CHECK ---")
    print(f"Topological Eccentricity (e):         {e:.6f}")
    print("---------------------------------------------------")
    print(f"Predicted Deflection Angle (ROM):     {delta_phi_rom_deg:.4f} deg")
    print(f"Observed Deflection Angle (JPL):      {delta_phi_obs_deg:.4f} deg")
    print("---------------------------------------------------")
    print(f"Angle Anomaly (Observed - ROM):       {angle_anomaly_deg:.4f} deg")

if __name__ == '__main__':
    run_rom_direct_angle_check()

Fetching JPL ephemeris (m, m/s) for Direct Angle Check...

--- R.O.M. PURE LOCAL DEFLECTION CHECK ---
Topological Eccentricity (e):         2.964090
---------------------------------------------------
Predicted Deflection Angle (ROM):     39.4337 deg
Observed Deflection Angle (JPL):      40.8877 deg
---------------------------------------------------
Angle Anomaly (Observed - ROM):       1.4540 deg


In [6]:
"""
WILL Relational Orbital Mechanics (R.O.M.)
Empirical Verification: 3-Body Topological Defect (Strategy A vs B)
Target: Juno Earth Flyby (Oct 9, 2013)
"""

import numpy as np
from astroquery.jplhorizons import Horizons
from astropy.time import Time

C_M_S = 299792458.0
AU_M = 149597870700.0
SEC_PER_DAY = 86400.0
R_S_EARTH_M = 0.008870056
R_S_SUN_M = 2953.25008

def get_state_vectors_si(target_id, center_id, epoch_str):
    jd = Time(epoch_str).jd
    obj = Horizons(id=target_id, location=center_id, epochs=jd)
    vec = obj.vectors()
    pos_m = np.array([vec['x'][0], vec['y'][0], vec['z'][0]]) * AU_M
    vel_ms = np.array([vec['vx'][0], vec['vy'][0], vec['vz'][0]]) * (AU_M / SEC_PER_DAY)
    return pos_m, vel_ms

def get_perigee_data(target_id, center_id, start, stop, step):
    obj = Horizons(id=target_id, location=center_id, epochs={'start': start, 'stop': stop, 'step': step})
    vec = obj.vectors()
    pos_m = np.column_stack((vec['x'], vec['y'], vec['z'])) * AU_M
    vel_ms = np.column_stack((vec['vx'], vec['vy'], vec['vz'])) * (AU_M / SEC_PER_DAY)
    distances = np.linalg.norm(pos_m, axis=1)
    min_idx = np.argmin(distances)
    return float(distances[min_idx]), vel_ms[min_idx]

def run_rom_3body_strategies():
    epoch_in = '2013-09-29 00:00:00'
    epoch_out = '2013-10-19 00:00:00'
    peri_start = '2013-10-09 18:00'
    peri_stop = '2013-10-09 21:00'

    print("Fetching JPL ephemeris for 3-Body Testing...")

    try:
        # Geocentric & Heliocentric boundaries
        _, vel_juno_earth_in_ms = get_state_vectors_si('-61', '@399', epoch_in)
        _, vel_juno_earth_out_obs_ms = get_state_vectors_si('-61', '@399', epoch_out)

        # Perigee state for 3-body coupling
        pos_earth_sun_peri_m, vel_earth_sun_peri_ms = get_state_vectors_si('399', '@10', '2013-10-09 19:21:00')
        r_p_m, vel_juno_earth_peri_ms = get_perigee_data('-61', '@399', peri_start, peri_stop, '1m')

    except Exception as e:
        print(f"API Error: {e}")
        return

    # --- KINEMATIC & POTENTIAL BASELINES ---
    beta_loc_in = np.linalg.norm(vel_juno_earth_in_ms) / C_M_S
    beta_E_peri = np.linalg.norm(vel_earth_sun_peri_ms) / C_M_S
    beta_pE = np.linalg.norm(vel_juno_earth_peri_ms) / C_M_S

    r_sun_m = np.linalg.norm(pos_earth_sun_peri_m)
    kappa_S_sq = R_S_SUN_M / r_sun_m
    kappa_pE_sq = R_S_EARTH_M / r_p_m
    beta_pE_sq = (beta_loc_in**2) + kappa_pE_sq

    # Phase Angle at Perigee (Earth's macro vector vs Probe's local perigee vector)
    unit_vE = vel_earth_sun_peri_ms / np.linalg.norm(vel_earth_sun_peri_ms)
    unit_vp = vel_juno_earth_peri_ms / np.linalg.norm(vel_juno_earth_peri_ms)
    cos_phi_p = np.dot(unit_vE, unit_vp)

    # --- BASELINE: 2-BODY ---
    delta_p_base = kappa_pE_sq / (2.0 * beta_pE_sq)
    e_base = (1.0 / delta_p_base) - 1.0
    phi_base_deg = np.degrees(2.0 * np.arcsin(1.0 / e_base))

    # --- STRATEGY A: Scalar Nesting ---
    kappa_eff_A_sq = kappa_pE_sq * (1.0 + (kappa_pE_sq / kappa_S_sq))
    delta_p_A = kappa_eff_A_sq / (2.0 * beta_pE_sq)
    e_A = (1.0 / delta_p_A) - 1.0
    phi_A_deg = np.degrees(2.0 * np.arcsin(1.0 / e_A))

    # --- STRATEGY B: Phase-Coupled Integration ---
    K_cross = 2.0 * beta_E_peri * beta_pE * cos_phi_p
    kappa_eff_B_sq = kappa_pE_sq + (kappa_pE_sq / kappa_S_sq) * K_cross
    delta_p_B = kappa_eff_B_sq / (2.0 * beta_pE_sq)
    e_B = (1.0 / delta_p_B) - 1.0
    phi_B_deg = np.degrees(2.0 * np.arcsin(1.0 / e_B))

    # --- OBSERVED JPL ---
    unit_in = vel_juno_earth_in_ms / np.linalg.norm(vel_juno_earth_in_ms)
    unit_out = vel_juno_earth_out_obs_ms / np.linalg.norm(vel_juno_earth_out_obs_ms)
    phi_obs_deg = np.degrees(np.arccos(np.clip(np.dot(unit_in, unit_out), -1.0, 1.0)))

    print("\n--- R.O.M. 3-BODY TOPOLOGICAL TESTING ---")
    print(f"Observed Deflection (JPL):  {phi_obs_deg:.4f} deg")
    print(f"Base 2-Body Deflection:     {phi_base_deg:.4f} deg (Error: {phi_obs_deg - phi_base_deg:.4f} deg)")
    print("---------------------------------------------------")
    print(f"Strategy A (Scalar):        {phi_A_deg:.4f} deg (Error: {phi_obs_deg - phi_A_deg:.4f} deg)")
    print(f"Strategy B (Phase-Coupled): {phi_B_deg:.4f} deg (Error: {phi_obs_deg - phi_B_deg:.4f} deg)")

if __name__ == '__main__':
    run_rom_3body_strategies()

Fetching JPL ephemeris for 3-Body Testing...

--- R.O.M. 3-BODY TOPOLOGICAL TESTING ---
Observed Deflection (JPL):  40.8877 deg
Base 2-Body Deflection:     39.4337 deg (Error: 1.4540 deg)
---------------------------------------------------
Strategy A (Scalar):        43.0846 deg (Error: -2.1970 deg)
Strategy B (Phase-Coupled): 59.4977 deg (Error: -18.6100 deg)


In [7]:
"""
WILL Relational Orbital Mechanics (R.O.M.)
Empirical Verification: 3-Body Superposition (Direct Addition vs Angle Summation)
Target: Juno Earth Flyby (Oct 9, 2013)
"""

import numpy as np
from astroquery.jplhorizons import Horizons
from astropy.time import Time

# --- STRICT SI CONSTANTS ---
C_M_S = 299792458.0
AU_M = 149597870700.0
SEC_PER_DAY = 86400.0
R_S_EARTH_M = 0.008870056
R_S_SUN_M = 2953.25008

def get_state_vectors_si(target_id, center_id, epoch_str):
    jd = Time(epoch_str).jd
    obj = Horizons(id=target_id, location=center_id, epochs=jd)
    vec = obj.vectors()
    pos_m = np.array([vec['x'][0], vec['y'][0], vec['z'][0]]) * AU_M
    vel_ms = np.array([vec['vx'][0], vec['vy'][0], vec['vz'][0]]) * (AU_M / SEC_PER_DAY)
    return pos_m, vel_ms

def get_perigee_data(target_id, center_id, start, stop, step):
    obj = Horizons(id=target_id, location=center_id, epochs={'start': start, 'stop': stop, 'step': step})
    vec = obj.vectors()
    pos_m = np.column_stack((vec['x'], vec['y'], vec['z'])) * AU_M
    distances = np.linalg.norm(pos_m, axis=1)
    return float(np.min(distances))

def calculate_angle(v1, v2):
    unit_v1 = v1 / np.linalg.norm(v1)
    unit_v2 = v2 / np.linalg.norm(v2)
    return np.arccos(np.clip(np.dot(unit_v1, unit_v2), -1.0, 1.0))

def run_rom_superposition_tests():
    epoch_in = '2013-09-29 00:00:00'
    epoch_out = '2013-10-19 00:00:00'
    peri_start = '2013-10-09 18:00'
    peri_stop = '2013-10-09 21:00'
    peri_exact = '2013-10-09 19:21:00'

    print("Fetching JPL ephemeris for Superposition Tests...")

    try:
        # Geocentric & Heliocentric vectors
        _, vel_juno_earth_in_ms = get_state_vectors_si('-61', '@399', epoch_in)
        _, vel_juno_earth_out_obs_ms = get_state_vectors_si('-61', '@399', epoch_out)

        pos_juno_sun_peri_m, vel_juno_sun_peri_ms = get_state_vectors_si('-61', '@10', peri_exact)
        r_pE_m = get_perigee_data('-61', '@399', peri_start, peri_stop, '1m')

    except Exception as e:
        print(f"API Error: {e}")
        return

    # --- BASE KINEMATICS & POTENTIALS ---
    beta_loc_in = np.linalg.norm(vel_juno_earth_in_ms) / C_M_S
    beta_sun_peri = np.linalg.norm(vel_juno_sun_peri_ms) / C_M_S

    kappa_pE_sq = R_S_EARTH_M / r_pE_m
    beta_pE_sq = (beta_loc_in**2) + kappa_pE_sq

    r_sun_m = np.linalg.norm(pos_juno_sun_peri_m)
    kappa_S_sq = R_S_SUN_M / r_sun_m

    # =========================================================
    # --- GROUND TRUTH & BASE 2-BODY ---
    # =========================================================

    delta_phi_obs_deg = np.degrees(calculate_angle(vel_juno_earth_in_ms, vel_juno_earth_out_obs_ms))

    e_base = (2.0 * beta_pE_sq / kappa_pE_sq) - 1.0
    phi_base_deg = np.degrees(2.0 * np.arcsin(1.0 / e_base))

    # =========================================================
    # --- METHOD 1: DIRECT ADDITION (NUMERATOR & DENOMINATOR) ---
    # =========================================================

    beta_combined_sq = beta_pE_sq + (beta_sun_peri**2)
    kappa_combined_sq = kappa_pE_sq + kappa_S_sq

    e_M1 = (2.0 * beta_combined_sq / kappa_combined_sq) - 1.0

    if e_M1 >= 1.0:
        phi_M1_deg = np.degrees(2.0 * np.arcsin(1.0 / e_M1))
    else:
        phi_M1_deg = float('nan') # Elliptical capture

    # =========================================================
    # --- METHOD 2: SEPARATE ANGLE CALCULATION ---
    # =========================================================

    # Calculate Sun's structural defect at the interaction distance
    e_Sun = (2.0 * (beta_sun_peri**2) / kappa_S_sq) - 1.0

    try:
        if e_Sun >= 1.0:
            phi_Sun_deg = np.degrees(2.0 * np.arcsin(1.0 / e_Sun))
            phi_M2_sum = phi_base_deg + phi_Sun_deg
        else:
            phi_Sun_deg = float('nan')
            phi_M2_sum = float('nan')
    except ValueError:
        phi_Sun_deg = float('nan')
        phi_M2_sum = float('nan')

    # --- TERMINAL OUTPUT ---
    print("\n--- R.O.M. 3-BODY SUPERPOSITION TESTS ---")
    print(f"Earth Local Potential (kappa_pE^2): {kappa_pE_sq:.8e}")
    print(f"Sun Macro Potential (kappa_S^2):    {kappa_S_sq:.8e}")
    print("---------------------------------------------------")
    print(f"Observed Deflection (JPL):          {delta_phi_obs_deg:.4f} deg")
    print(f"Base 2-Body Deflection (ROM):       {phi_base_deg:.4f} deg (Error: {delta_phi_obs_deg - phi_base_deg:.4f} deg)")
    print("---------------------------------------------------")
    print(f"METHOD 1: Direct Addition (e_M1):   {e_M1:.6f}")
    if np.isnan(phi_M1_deg):
        print("METHOD 1 Result:                    FAILED (e < 1, Elliptical regime)")
    else:
        print(f"METHOD 1 Result:                    {phi_M1_deg:.4f} deg (Error: {delta_phi_obs_deg - phi_M1_deg:.4f} deg)")
    print("---------------------------------------------------")
    print(f"METHOD 2: Sun-Only Defect (e_Sun):  {e_Sun:.6f}")
    if np.isnan(phi_Sun_deg):
        print("METHOD 2 Result:                    FAILED (e_Sun < 1, Probe is bound to Sun)")
    else:
        print(f"METHOD 2 Result (Summation):        {phi_M2_sum:.4f} deg")

if __name__ == '__main__':
    run_rom_superposition_tests()

Fetching JPL ephemeris for Superposition Tests...

--- R.O.M. 3-BODY SUPERPOSITION TESTS ---
Earth Local Potential (kappa_pE^2): 1.27777167e-09
Sun Macro Potential (kappa_S^2):    1.97662512e-08
---------------------------------------------------
Observed Deflection (JPL):          40.8877 deg
Base 2-Body Deflection (ROM):       39.4337 deg (Error: 1.4540 deg)
---------------------------------------------------
METHOD 1: Direct Addition (e_M1):   0.962826
METHOD 1 Result:                    FAILED (e < 1, Elliptical regime)
---------------------------------------------------
METHOD 2: Sun-Only Defect (e_Sun):  0.833456
METHOD 2 Result:                    FAILED (e_Sun < 1, Probe is bound to Sun)


In [8]:
"""
WILL Relational Orbital Mechanics (R.O.M.)
Empirical Verification: Ultra-Narrow Window Isolation
Target: Juno Earth Flyby (Oct 8 - Oct 10, 2013)
"""

import numpy as np
from astroquery.jplhorizons import Horizons
from astropy.time import Time

C_M_S = 299792458.0
AU_M = 149597870700.0
SEC_PER_DAY = 86400.0
R_S_EARTH_M = 0.008870056

def get_state_vectors_si(target_id, center_id, epoch_str):
    jd = Time(epoch_str).jd
    obj = Horizons(id=target_id, location=center_id, epochs=jd)
    vec = obj.vectors()
    pos_m = np.array([vec['x'][0], vec['y'][0], vec['z'][0]]) * AU_M
    vel_ms = np.array([vec['vx'][0], vec['vy'][0], vec['vz'][0]]) * (AU_M / SEC_PER_DAY)
    return pos_m, vel_ms

def get_perigee_meters(target_id, center_id, start, stop, step):
    obj = Horizons(id=target_id, location=center_id, epochs={'start': start, 'stop': stop, 'step': step})
    vec = obj.vectors()
    pos_m = np.column_stack((vec['x'], vec['y'], vec['z'])) * AU_M
    return float(np.min(np.linalg.norm(pos_m, axis=1)))

def calculate_angle(v1, v2):
    unit_v1 = v1 / np.linalg.norm(v1)
    unit_v2 = v2 / np.linalg.norm(v2)
    return np.arccos(np.clip(np.dot(unit_v1, unit_v2), -1.0, 1.0))

def run_rom_narrow_angle_check():
    # ULTRA-NARROW WINDOW: +/- 24 Hours from Perigee
    # Isolates Earth's topological defect from the Sun's macro-gradient.
    epoch_in = '2013-10-08 19:00:00'
    epoch_out = '2013-10-10 19:00:00'
    peri_start = '2013-10-09 18:00'
    peri_stop = '2013-10-09 21:00'

    print("Fetching JPL ephemeris for Ultra-Narrow Angle Check...")

    try:
        # GEOCENTRIC ONLY (Probe w.r.t Earth)
        _, vel_juno_earth_in_ms = get_state_vectors_si('-61', '@399', epoch_in)
        _, vel_juno_earth_out_obs_ms = get_state_vectors_si('-61', '@399', epoch_out)

        r_p_m = get_perigee_meters('-61', '@399', peri_start, peri_stop, '1m')

    except Exception as e:
        print(f"API Error: {e}")
        return

    # 1. R.O.M. ALGEBRAIC PREDICTION (Infinite Asymptote)
    beta_loc_in = np.linalg.norm(vel_juno_earth_in_ms) / C_M_S
    kappa_pE_sq = R_S_EARTH_M / r_p_m
    beta_pE_sq = (beta_loc_in**2) + kappa_pE_sq

    delta_p = kappa_pE_sq / (2.0 * beta_pE_sq)
    e = (1.0 / delta_p) - 1.0

    # Pure algebraic topological deflection (Theoretical Max)
    delta_phi_rom_rad = 2.0 * np.arcsin(1.0 / e)
    delta_phi_rom_deg = np.degrees(delta_phi_rom_rad)

    # 2. JPL OBSERVED DEFLECTION (Truncated Ground Truth)
    delta_phi_obs_rad = calculate_angle(vel_juno_earth_in_ms, vel_juno_earth_out_obs_ms)
    delta_phi_obs_deg = np.degrees(delta_phi_obs_rad)

    # 3. ANALYSIS
    print("\n--- R.O.M. ULTRA-NARROW ISOLATION CHECK (+/- 24H) ---")
    print(f"Topological Eccentricity (e):         {e:.6f}")
    print("---------------------------------------------------")
    print(f"Theoretical Max Deflection (ROM):     {delta_phi_rom_deg:.4f} deg")
    print(f"Truncated Deflection (JPL):           {delta_phi_obs_deg:.4f} deg")
    print("---------------------------------------------------")
    print(f"Angle Deficit (ROM - JPL):            {delta_phi_rom_deg - delta_phi_obs_deg:.4f} deg")

if __name__ == '__main__':
    run_rom_narrow_angle_check()

Fetching JPL ephemeris for Ultra-Narrow Angle Check...

--- R.O.M. ULTRA-NARROW ISOLATION CHECK (+/- 24H) ---
Topological Eccentricity (e):         2.894295
---------------------------------------------------
Theoretical Max Deflection (ROM):     40.4255 deg
Truncated Deflection (JPL):           40.6677 deg
---------------------------------------------------
Angle Deficit (ROM - JPL):            -0.2422 deg


In [9]:
"""
WILL Relational Orbital Mechanics (R.O.M.)
Empirical Verification: Dynamic Topological Handoff (Gradient Balance)
Target: Juno Earth Flyby (Oct 2013)
"""

import numpy as np
from astroquery.jplhorizons import Horizons

# --- STRICT SI CONSTANTS ---
C_M_S = 299792458.0
AU_M = 149597870700.0
SEC_PER_DAY = 86400.0
R_S_EARTH_M = 0.008870056
R_S_SUN_M = 2953.25008

def fetch_jpl_trajectory(target_id, center_id, start, stop, step):
    """Fetches full trajectory array strictly in meters and meters/second."""
    obj = Horizons(id=target_id, location=center_id, epochs={'start': start, 'stop': stop, 'step': step})
    vec = obj.vectors()

    times = vec['datetime_jd']
    times_str = vec['datetime_str']

    pos_m = np.column_stack((vec['x'], vec['y'], vec['z'])) * AU_M
    vel_ms = np.column_stack((vec['vx'], vec['vy'], vec['vz'])) * (AU_M / SEC_PER_DAY)

    return times, times_str, pos_m, vel_ms

def calculate_angle(v1, v2):
    unit_v1 = v1 / np.linalg.norm(v1)
    unit_v2 = v2 / np.linalg.norm(v2)
    return np.arccos(np.clip(np.dot(unit_v1, unit_v2), -1.0, 1.0))

def run_rom_topological_handoff():
    # Wide search window to allow dynamic boundary detection
    search_start = '2013-10-01'
    search_stop = '2013-10-18'
    search_step = '1h'

    print("Fetching 1-hour JPL ephemeris for Gradient Analysis...")

    try:
        _, t_str, pos_juno_earth, vel_juno_earth = fetch_jpl_trajectory('-61', '@399', search_start, search_stop, search_step)
        _, _, pos_juno_sun, vel_juno_sun = fetch_jpl_trajectory('-61', '@10', search_start, search_stop, search_step)
        _, _, pos_earth_sun, vel_earth_sun = fetch_jpl_trajectory('399', '@10', search_start, search_stop, search_step)
    except Exception as e:
        print(f"API Error: {e}")
        return

    # =========================================================
    # --- 1. DYNAMIC BOUNDARY DETECTION (GRADIENT BALANCE) ---
    # =========================================================

    r_juno_earth = np.linalg.norm(pos_juno_earth, axis=1)
    r_juno_sun = np.linalg.norm(pos_juno_sun, axis=1)

    kappa_E_sq = R_S_EARTH_M / r_juno_earth
    kappa_S_sq = R_S_SUN_M / r_juno_sun

    # Discrete gradients (absolute rate of change of potential capacity)
    d_kappa_E_sq = np.abs(np.diff(kappa_E_sq))
    d_kappa_S_sq = np.abs(np.diff(kappa_S_sq))

    # Boolean array: True when Earth's structural pull changes faster than Sun's
    earth_dominant = d_kappa_E_sq > d_kappa_S_sq

    # Find Perigee index to split Inbound and Outbound search
    idx_perigee = np.argmin(r_juno_earth)

    # Find Inbound Boundary (first moment Earth dominates)
    # Search forward from start
    idx_in = 0
    for i in range(idx_perigee):
        if earth_dominant[i]:
            idx_in = i
            break

    # Find Outbound Boundary (first moment Sun re-dominates after perigee)
    # Search forward from perigee
    idx_out = len(earth_dominant) - 1
    for i in range(idx_perigee, len(earth_dominant)):
        if not earth_dominant[i]:
            idx_out = i
            break

    print(f"\n--- TOPOLOGICAL BUBBLE BOUNDARIES ---")
    print(f"Handoff IN:  {t_str[idx_in]}")
    print(f"Perigee:     {t_str[idx_perigee]}")
    print(f"Handoff OUT: {t_str[idx_out]}")
    print(f"Window Size: {(idx_out - idx_in)} hours")

    # =========================================================
    # --- 2. R.O.M. 2-BODY ISOLATION (EARTH FRAME) ---
    # =========================================================

    beta_loc_in = np.linalg.norm(vel_juno_earth[idx_in]) / C_M_S
    r_p_m = r_juno_earth[idx_perigee]

    kappa_pE_sq = R_S_EARTH_M / r_p_m
    beta_pE_sq = (beta_loc_in**2) + kappa_pE_sq

    delta_p = kappa_pE_sq / (2.0 * beta_pE_sq)
    e = (1.0 / delta_p) - 1.0
    delta_phi = 2.0 * np.arcsin(1.0 / e)

    # =========================================================
    # --- 3. SPHERICAL PHASE CLOSURE (AT HANDOFF OUT) ---
    # =========================================================

    v_loc_in = vel_juno_earth[idx_in]
    v_E_out = vel_earth_sun[idx_out]
    p_loc_in = pos_juno_earth[idx_in]

    phi_ref = calculate_angle(v_loc_in, v_E_out)

    n_orb = np.cross(p_loc_in, v_loc_in)
    n_ref = np.cross(v_loc_in, v_E_out)
    cos_theta = np.dot(n_orb, n_ref) / (np.linalg.norm(n_orb) * np.linalg.norm(n_ref))

    cos_phi_out = (np.cos(phi_ref) * np.cos(delta_phi)) + (np.sin(phi_ref) * np.sin(delta_phi) * cos_theta)
    phi_out = np.arccos(cos_phi_out)

    # =========================================================
    # --- 4. MACRO-ASSEMBLY (SUN FRAME RE-ENTRY) ---
    # =========================================================

    beta_E_out = np.linalg.norm(v_E_out) / C_M_S

    # Global superposition using preserved local kinetic buffer and new spherical phase
    beta_Z_out_pred_sq = (beta_E_out**2) + (beta_loc_in**2) + (2.0 * beta_E_out * beta_loc_in * cos_phi_out)

    v_out_pred_ms = C_M_S * np.sqrt(beta_Z_out_pred_sq)
    v_out_obs_ms = np.linalg.norm(vel_juno_sun[idx_out])

    anomaly_mms = (v_out_obs_ms - v_out_pred_ms) * 1000.0

    print("\n--- R.O.M. DISCRETE HANDOFF RESULTS ---")
    print(f"Topological Eccentricity (e):         {e:.6f}")
    print(f"Phase Deflection Arc (delta_phi):     {np.degrees(delta_phi):.4f} deg")
    print("---------------------------------------------------")
    print(f"Observed Terminal Velocity (JPL):     {v_out_obs_ms / 1000.0:.6f} km/s")
    print(f"Predicted Terminal Velocity (ROM):    {v_out_pred_ms / 1000.0:.6f} km/s")
    print("---------------------------------------------------")
    print(f"Velocity Anomaly (Observed - ROM):    {anomaly_mms:.4f} mm/s")

if __name__ == '__main__':
    run_rom_topological_handoff()

Fetching 1-hour JPL ephemeris for Gradient Analysis...

--- TOPOLOGICAL BUBBLE BOUNDARIES ---
Handoff IN:  A.D. 2013-Oct-09 12:00:00.0000
Perigee:     A.D. 2013-Oct-09 19:00:00.0000
Handoff OUT: A.D. 2013-Oct-10 05:00:00.0000
Window Size: 17 hours

--- R.O.M. DISCRETE HANDOFF RESULTS ---
Topological Eccentricity (e):         6.024500
Phase Deflection Arc (delta_phi):     19.1094 deg
---------------------------------------------------
Observed Terminal Velocity (JPL):     38.796106 km/s
Predicted Terminal Velocity (ROM):    37.094570 km/s
---------------------------------------------------
Velocity Anomaly (Observed - ROM):    1701535.8664 mm/s


In [11]:
"""
WILL Relational Orbital Mechanics (R.O.M.)
Empirical Verification: Continuous Differential Phase Integration (3-Body)
Target: Juno Earth Flyby (Sept 29 - Oct 19, 2013)
"""

import numpy as np
from astroquery.jplhorizons import Horizons
from astropy.time import Time

# --- STRICT SI CONSTANTS ---
C_M_S = 299792458.0
AU_M = 149597870700.0
SEC_PER_DAY = 86400.0
R_S_EARTH_M = 0.008870056
R_S_SUN_M = 2953.25008

def fetch_jpl_trajectory(target_id, center_id, start, stop, step):
    """Fetches full trajectory array strictly in meters and meters/second."""
    obj = Horizons(id=target_id, location=center_id, epochs={'start': start, 'stop': stop, 'step': step})
    vec = obj.vectors()

    jd = vec['datetime_jd']
    pos_m = np.column_stack((vec['x'], vec['y'], vec['z'])) * AU_M
    vel_ms = np.column_stack((vec['vx'], vec['vy'], vec['vz'])) * (AU_M / SEC_PER_DAY)

    return jd, pos_m, vel_ms

def calculate_angle(v1, v2):
    unit_v1 = v1 / np.linalg.norm(v1)
    unit_v2 = v2 / np.linalg.norm(v2)
    return np.arccos(np.clip(np.dot(unit_v1, unit_v2), -1.0, 1.0))

def run_rom_continuous_integration():
    # Full 20-day window
    t_start = '2013-09-29 00:00:00'
    t_stop = '2013-10-19 00:00:00'
    t_step = '10m'  # High resolution for precise numerical integration

    print(f"Fetching {t_step} JPL ephemeris for Continuous Integration...")

    try:
        jd, pos_PE, vel_PE = fetch_jpl_trajectory('-61', '@399', t_start, t_stop, t_step)
        _, pos_PS, _ = fetch_jpl_trajectory('-61', '@10', t_start, t_stop, t_step)
        _, pos_ES, _ = fetch_jpl_trajectory('399', '@10', t_start, t_stop, t_step)
    except Exception as e:
        print(f"API Error: {e}")
        return

    # =========================================================
    # --- 1. BASELINE GEOMETRY ---
    # =========================================================

    # Establish the invariant orbital plane normal (using Perigee state)
    idx_p = np.argmin(np.linalg.norm(pos_PE, axis=1))
    n_vec = np.cross(pos_PE[idx_p], vel_PE[idx_p])
    n_hat = n_vec / np.linalg.norm(n_vec)

    # Ground Truth JPL Deflection
    v_in_obs = vel_PE[0]
    v_out_obs = vel_PE[-1]
    phi_obs_deg = np.degrees(calculate_angle(v_in_obs, v_out_obs))

    # Analytical 2-Body Expectation (from earlier tests)
    r_p = np.linalg.norm(pos_PE[idx_p])
    beta_loc_in = np.linalg.norm(v_in_obs) / C_M_S
    kappa_pE_sq = R_S_EARTH_M / r_p
    e_base = (2.0 * ((beta_loc_in**2) + kappa_pE_sq) / kappa_pE_sq) - 1.0
    phi_base_deg = np.degrees(2.0 * np.arcsin(1.0 / e_base))

    # =========================================================
    # --- 2. CONTINUOUS DIFFERENTIAL INTEGRATION ---
    # =========================================================

    total_dphi_E_rad = 0.0
    total_dphi_S_rad = 0.0

    print("Integrating R.O.M. differential equations...")

    # Iterate through all steps
    for i in range(len(jd) - 1):
        dt_sec = (jd[i+1] - jd[i]) * SEC_PER_DAY

        v_vec = vel_PE[i]
        v_mag = np.linalg.norm(v_vec)
        v_hat = v_vec / v_mag

        beta2 = (v_mag / C_M_S)**2
        ds = v_mag * dt_sec

        # -----------------------------------------------------
        # MICRO-NODE 1: EARTH'S TOPOLOGICAL PULL
        # -----------------------------------------------------
        # pos_PE is Earth->Probe. Probe is origin, so Earth is at -pos_PE
        r_E_vec = -pos_PE[i]
        r_E = np.linalg.norm(r_E_vec)
        u_E = r_E_vec / r_E

        # Structural Gradient = kappa^2 / r
        Gamma_E = (R_S_EARTH_M / r_E) / r_E
        G_E_vec = Gamma_E * u_E

        # -----------------------------------------------------
        # MICRO-NODE 2: SUN'S RELATIONAL TIDAL PULL
        # -----------------------------------------------------
        # Sun pull on Probe
        r_PS_vec = -pos_PS[i]
        r_PS = np.linalg.norm(r_PS_vec)
        u_PS = r_PS_vec / r_PS
        Gamma_S_P = (R_S_SUN_M / r_PS) / r_PS
        G_S_P_vec = Gamma_S_P * u_PS

        # Sun pull on Earth
        r_ES_vec = -pos_ES[i]
        r_ES = np.linalg.norm(r_ES_vec)
        u_ES = r_ES_vec / r_ES
        Gamma_S_E = (R_S_SUN_M / r_ES) / r_ES
        G_S_E_vec = Gamma_S_E * u_ES

        # True Relational Gradient = Gradient on Probe - Gradient on Earth
        G_S_tidal_vec = G_S_P_vec - G_S_E_vec

        # -----------------------------------------------------
        # KINEMATIC PROJECTIONS (LATERAL BENDING)
        # -----------------------------------------------------
        # Extract the component of the gradient perpendicular to velocity
        # lying strictly in the orbital plane (using cross product dotted with normal)
        lat_G_E = np.dot(np.cross(v_hat, G_E_vec), n_hat)
        lat_G_S = np.dot(np.cross(v_hat, G_S_tidal_vec), n_hat)

        # Differential Phase Curvature equation: dphi = (Gamma_lat / beta^2) * ds
        dphi_E = (lat_G_E / 2*beta2) * ds
        dphi_S = (lat_G_S / 2*beta2) * ds

        total_dphi_E_rad += dphi_E
        total_dphi_S_rad += dphi_S

    # =========================================================
    # --- 3. FINAL SYNTHESIS ---
    # =========================================================

    integrated_E_deg = np.degrees(total_dphi_E_rad)
    integrated_S_deg = np.degrees(total_dphi_S_rad)
    integrated_Total_deg = integrated_E_deg + integrated_S_deg

    anomaly_deg = phi_obs_deg - integrated_Total_deg

    print("\n--- R.O.M. CONTINUOUS DIFFERENTIAL SYNTHESIS ---")
    print(f"JPL Observed Geocentric Deflection:  {phi_obs_deg:.4f} deg")
    print("---------------------------------------------------")
    print(f"Integrated Earth Defect:             {integrated_E_deg:.4f} deg")
    print(f"Integrated Sun Tidal Perturbation:   {integrated_S_deg:.4f} deg")
    print("---------------------------------------------------")
    print(f"Total Integrated R.O.M. Deflection:  {integrated_Total_deg:.4f} deg")
    print(f"Angle Anomaly (Observed - ROM):      {anomaly_deg:.4f} deg")

if __name__ == '__main__':
    run_rom_continuous_integration()

Fetching 10m JPL ephemeris for Continuous Integration...
Integrating R.O.M. differential equations...

--- R.O.M. CONTINUOUS DIFFERENTIAL SYNTHESIS ---
JPL Observed Geocentric Deflection:  40.8877 deg
---------------------------------------------------
Integrated Earth Defect:             0.0000 deg
Integrated Sun Tidal Perturbation:   0.3675 deg
---------------------------------------------------
Total Integrated R.O.M. Deflection:  0.3675 deg
Angle Anomaly (Observed - ROM):      40.5201 deg


In [12]:
"""
WILL Relational Orbital Mechanics (R.O.M.)
Empirical Verification: Continuous Differential Phase Integration (3-Body)
Target: Juno Earth Flyby (Sept 29 - Oct 19, 2013)
Correction: Exact 2*beta^2 kinematic resistance & 1-minute resolution.
"""

import numpy as np
from astroquery.jplhorizons import Horizons
from astropy.time import Time

# --- STRICT SI CONSTANTS ---
C_M_S = 299792458.0
AU_M = 149597870700.0
SEC_PER_DAY = 86400.0
R_S_EARTH_M = 0.008870056
R_S_SUN_M = 2953.25008

def fetch_jpl_trajectory(target_id, center_id, start, stop, step):
    """Fetches full trajectory array strictly in meters and meters/second."""
    obj = Horizons(id=target_id, location=center_id, epochs={'start': start, 'stop': stop, 'step': step})
    vec = obj.vectors()

    jd = vec['datetime_jd']
    pos_m = np.column_stack((vec['x'], vec['y'], vec['z'])) * AU_M
    vel_ms = np.column_stack((vec['vx'], vec['vy'], vec['vz'])) * (AU_M / SEC_PER_DAY)

    return jd, pos_m, vel_ms

def calculate_angle(v1, v2):
    unit_v1 = v1 / np.linalg.norm(v1)
    unit_v2 = v2 / np.linalg.norm(v2)
    return np.arccos(np.clip(np.dot(unit_v1, unit_v2), -1.0, 1.0))

def run_rom_continuous_integration_corrected():
    # Full 20-day window
    t_start = '2013-09-29 00:00:00'
    t_stop = '2013-10-19 00:00:00'
    t_step = '1m'  # Max resolution for Euler integration

    print(f"Fetching {t_step} JPL ephemeris for Continuous Integration...")

    try:
        jd, pos_PE, vel_PE = fetch_jpl_trajectory('-61', '@399', t_start, t_stop, t_step)
        _, pos_PS, _ = fetch_jpl_trajectory('-61', '@10', t_start, t_stop, t_step)
        _, pos_ES, _ = fetch_jpl_trajectory('399', '@10', t_start, t_stop, t_step)
    except Exception as e:
        print(f"API Error: {e}")
        return

    # =========================================================
    # --- 1. BASELINE GEOMETRY ---
    # =========================================================

    # Establish the invariant orbital plane normal (using Perigee state)
    idx_p = np.argmin(np.linalg.norm(pos_PE, axis=1))
    n_vec = np.cross(pos_PE[idx_p], vel_PE[idx_p])
    n_hat = n_vec / np.linalg.norm(n_vec)

    # Ground Truth JPL Deflection
    v_in_obs = vel_PE[0]
    v_out_obs = vel_PE[-1]
    phi_obs_deg = np.degrees(calculate_angle(v_in_obs, v_out_obs))

    # Analytical 2-Body Expectation (Topological Baseline)
    r_p = np.linalg.norm(pos_PE[idx_p])
    beta_loc_in = np.linalg.norm(v_in_obs) / C_M_S
    kappa_pE_sq = R_S_EARTH_M / r_p
    e_base = (2.0 * ((beta_loc_in**2) + kappa_pE_sq) / kappa_pE_sq) - 1.0
    phi_base_deg = np.degrees(2.0 * np.arcsin(1.0 / e_base))

    # =========================================================
    # --- 2. CONTINUOUS DIFFERENTIAL INTEGRATION ---
    # =========================================================

    total_dphi_E_rad = 0.0
    total_dphi_S_rad = 0.0

    print("Integrating strict R.O.M. differential equations (2*beta^2 invariant)...")

    # Iterate through all steps
    for i in range(len(jd) - 1):
        dt_sec = (jd[i+1] - jd[i]) * SEC_PER_DAY

        v_vec = vel_PE[i]
        v_mag = np.linalg.norm(v_vec)
        v_hat = v_vec / v_mag

        beta2 = (v_mag / C_M_S)**2
        ds = v_mag * dt_sec

        # -----------------------------------------------------
        # MICRO-NODE 1: EARTH'S TOPOLOGICAL PULL
        # -----------------------------------------------------
        r_E_vec = -pos_PE[i]
        r_E = np.linalg.norm(r_E_vec)
        u_E = r_E_vec / r_E

        Gamma_E = (R_S_EARTH_M / r_E) / r_E
        G_E_vec = Gamma_E * u_E

        # -----------------------------------------------------
        # MICRO-NODE 2: SUN'S RELATIONAL TIDAL PULL
        # -----------------------------------------------------
        r_PS_vec = -pos_PS[i]
        r_PS = np.linalg.norm(r_PS_vec)
        u_PS = r_PS_vec / r_PS
        Gamma_S_P = (R_S_SUN_M / r_PS) / r_PS
        G_S_P_vec = Gamma_S_P * u_PS

        r_ES_vec = -pos_ES[i]
        r_ES = np.linalg.norm(r_ES_vec)
        u_ES = r_ES_vec / r_ES
        Gamma_S_E = (R_S_SUN_M / r_ES) / r_ES
        G_S_E_vec = Gamma_S_E * u_ES

        # Relational Gradient (Difference in structural pull)
        G_S_tidal_vec = G_S_P_vec - G_S_E_vec

        # -----------------------------------------------------
        # KINEMATIC PROJECTIONS (LATERAL BENDING)
        # -----------------------------------------------------
        lat_G_E = np.dot(np.cross(v_hat, G_E_vec), n_hat)
        lat_G_S = np.dot(np.cross(v_hat, G_S_tidal_vec), n_hat)

        # STRICT ROM DEFORMATION EQUATION: Denominator is strictly 2 * beta^2
        dphi_E = (lat_G_E / (2.0 * beta2)) * ds
        dphi_S = (lat_G_S / (2.0 * beta2)) * ds

        total_dphi_E_rad += dphi_E
        total_dphi_S_rad += dphi_S

    # =========================================================
    # --- 3. FINAL SYNTHESIS ---
    # =========================================================

    integrated_E_deg = np.degrees(total_dphi_E_rad)
    integrated_S_deg = np.degrees(total_dphi_S_rad)
    integrated_Total_deg = integrated_E_deg + integrated_S_deg

    anomaly_deg = phi_obs_deg - integrated_Total_deg

    print("\n--- R.O.M. CONTINUOUS DIFFERENTIAL SYNTHESIS ---")
    print(f"JPL Observed Geocentric Deflection:  {phi_obs_deg:.4f} deg")
    print(f"Static 2-Body Baseline Expectation:  {phi_base_deg:.4f} deg")
    print("---------------------------------------------------")
    print(f"Integrated Earth Defect:             {integrated_E_deg:.4f} deg")
    print(f"Integrated Sun Tidal Perturbation:   {integrated_S_deg:.4f} deg")
    print("---------------------------------------------------")
    print(f"Total Integrated R.O.M. Deflection:  {integrated_Total_deg:.4f} deg")
    print(f"Angle Anomaly (Observed - ROM):      {anomaly_deg:.4f} deg")

if __name__ == '__main__':
    run_rom_continuous_integration_corrected()

Fetching 1m JPL ephemeris for Continuous Integration...
Integrating strict R.O.M. differential equations (2*beta^2 invariant)...

--- R.O.M. CONTINUOUS DIFFERENTIAL SYNTHESIS ---
JPL Observed Geocentric Deflection:  40.8877 deg
Static 2-Body Baseline Expectation:  39.4337 deg
---------------------------------------------------
Integrated Earth Defect:             40.6772 deg
Integrated Sun Tidal Perturbation:   0.1823 deg
---------------------------------------------------
Total Integrated R.O.M. Deflection:  40.8595 deg
Angle Anomaly (Observed - ROM):      0.0282 deg
